In [1]:
import pandas as pd

rpe1_fitness = pd.read_csv(
    "../data/raw/rpe1_hart2015_orcs_matrix.tsv",
    sep="\t",
    na_values="-",
)

print(rpe1_fitness.shape)
print(rpe1_fitness.columns.tolist())

rpe1_fitness[
    ["OFFICIAL_SYMBOL", "28-SCORE.1", "28-HIT"]
].head()

(17203, 12)
['IDENTIFIER_ID', 'IDENTIFIER_TYPE', 'OFFICIAL_SYMBOL', 'ALIASES', 'ORGANISM_ID', 'ORGANISM_OFFICIAL', '28-SCORE.1', '28-SCORE.2', '28-SCORE.3', '28-SCORE.4', '28-SCORE.5', '28-HIT']


,OFFICIAL_SYMBOL,28-SCORE.1,28-HIT
0,GPRIN1,-49.529,NO
1,XKR4,-42.628,NO
2,MBD6,-8.458,NO
3,CSMD2,-36.005,NO
4,LMTK3,-45.181,NO


In [2]:
print(
    rpe1_fitness["28-SCORE.1"].notna().sum()
)

print(
    rpe1_fitness["OFFICIAL_SYMBOL"].nunique()
)

print(
    rpe1_fitness["28-SCORE.1"].describe()
)

17203
17203
count    17203.000000
mean       -20.029817
std         29.484871
min        -63.793000
25%        -39.537500
50%        -25.842000
75%         -9.616000
max        152.816000
Name: 28-SCORE.1, dtype: float64


In [3]:
import scanpy as sc

rpe1 = sc.read_h5ad(
    "../data/processed/replogle22rpe1.h5ad",
    backed="r",
)

In [4]:
rpe1_targets = (
    rpe1.obs["perturbation"]
    .astype(str)
    .unique()
    .tolist()
)

rpe1_targets = [
    gene
    for gene in rpe1_targets
    if gene != "control"
]

print(
    "RPE1 Perturb-seq targets:",
    len(rpe1_targets)
)

RPE1 Perturb-seq targets: 2016


In [5]:
fitness_genes = set(
    rpe1_fitness["OFFICIAL_SYMBOL"]
    .astype(str)
)

rpe1_fitness_targets = [
    gene
    for gene in rpe1_targets
    if gene in fitness_genes
]

missing_fitness = [
    gene
    for gene in rpe1_targets
    if gene not in fitness_genes
]

print(
    "Perturb-seq targets:",
    len(rpe1_targets)
)

print(
    "Targets with RPE1 fitness:",
    len(rpe1_fitness_targets)
)

print(
    "Coverage:",
    len(rpe1_fitness_targets)
    / len(rpe1_targets)
)

print(
    "Missing fitness targets:",
    len(missing_fitness)
)

print(
    "Examples missing:",
    missing_fitness[:20]
)

Perturb-seq targets: 2016
Targets with RPE1 fitness: 1765
Coverage: 0.8754960317460317
Missing fitness targets: 251
Examples missing: ['ABCE1', 'AC118549.1', 'ACTR2', 'ALG1L', 'ANAPC1', 'ARGFX', 'ARHGAP11B', 'ARPC3', 'ATF4', 'ATP5F1A', 'ATP5F1B', 'ATP5F1C', 'ATP5ME', 'ATP5MF', 'ATP5PD', 'ATP5PO', 'BANF1', 'BCLAF1', 'BET1', 'BTF3L4']


In [6]:
k562 = sc.read_h5ad(
    "../data/processed/replogle22k562.h5ad",
    backed="r",
)

In [7]:
k562_targets = set(
    k562.obs["perturbation"]
    .astype(str)
    .unique()
)

k562_targets.discard("control")

rpe1_target_set = set(
    rpe1_targets
)

shared_targets = [
    gene
    for gene in rpe1_targets
    if gene in k562_targets
]

shared_with_rpe1_fitness = [
    gene
    for gene in shared_targets
    if gene in fitness_genes
]

print(
    "Shared K562/RPE1 perturbations:",
    len(shared_targets)
)

print(
    "Shared targets with RPE1 fitness:",
    len(shared_with_rpe1_fitness)
)

print(
    "Coverage of shared universe:",
    len(shared_with_rpe1_fitness)
    / len(shared_targets)
)

Shared K562/RPE1 perturbations: 1676
Shared targets with RPE1 fitness: 1467
Coverage of shared universe: 0.8752983293556086


In [8]:
rpe1_gene_fitness = (
    rpe1_fitness[
        [
            "OFFICIAL_SYMBOL",
            "28-SCORE.1",
        ]
    ]
    .rename(
        columns={
            "OFFICIAL_SYMBOL": "gene",
            "28-SCORE.1": "rpe1_fitness",
        }
    )
    .copy()
)

In [9]:
rpe1_gene_fitness.head()
rpe1_gene_fitness["rpe1_fitness"].describe()

count    17203.000000
mean       -20.029817
std         29.484871
min        -63.793000
25%        -39.537500
50%        -25.842000
75%         -9.616000
max        152.816000
Name: rpe1_fitness, dtype: float64

In [10]:
import pandas as pd


# Exact official-symbol mapping
official_to_score = (
    rpe1_fitness
    .set_index("OFFICIAL_SYMBOL")["28-SCORE.1"]
    .to_dict()
)


# Build alias -> official-symbol mapping
alias_to_officials = {}

for _, row in rpe1_fitness.iterrows():

    official = str(row["OFFICIAL_SYMBOL"])

    aliases = row["ALIASES"]

    if pd.isna(aliases):
        continue

    for alias in str(aliases).split("|"):

        alias = alias.strip()

        if not alias:
            continue

        alias_to_officials.setdefault(
            alias,
            set(),
        ).add(official)

In [11]:
unique_alias_to_official = {
    alias: next(iter(officials))
    for alias, officials
    in alias_to_officials.items()
    if len(officials) == 1
}

In [12]:
alias_matches = {
    target: unique_alias_to_official[target]
    for target in missing_fitness
    if target in unique_alias_to_official
}

print(
    "Missing before alias matching:",
    len(missing_fitness)
)

print(
    "Recovered by unique alias:",
    len(alias_matches)
)

print(
    "Still missing:",
    len(missing_fitness)
    - len(alias_matches)
)

list(alias_matches.items())[:30]

Missing before alias matching: 251
Recovered by unique alias: 9
Still missing: 242


[('GCOM1', 'MYZAP'),
 ('GET1', 'WRB'),
 ('GET3', 'ASNA1'),
 ('MST1', 'STK4'),
 ('POGLUT3', 'KDELC2'),
 ('PRORP', 'KIAA0391'),
 ('RPL23', 'MRPL23'),
 ('SNRPD1', 'SNRPD2'),
 ('TXN', 'TXN2')]

In [13]:
rows = []

for target in rpe1_targets:

    if target in official_to_score:

        rows.append({
            "gene": target,
            "orcs_gene": target,
            "rpe1_fitness": official_to_score[target],
            "match_type": "official_symbol",
        })

    elif target in unique_alias_to_official:

        official = unique_alias_to_official[target]

        rows.append({
            "gene": target,
            "orcs_gene": official,
            "rpe1_fitness": official_to_score[official],
            "match_type": "unique_alias",
        })


rpe1_gene_fitness = pd.DataFrame(rows)

In [14]:
print(rpe1_gene_fitness.shape)

print(
    rpe1_gene_fitness["match_type"]
    .value_counts()
)

print(
    rpe1_gene_fitness["gene"]
    .nunique()
)

rpe1_gene_fitness["rpe1_fitness"].describe()

(1774, 4)
match_type
official_symbol    1765
unique_alias          9
Name: count, dtype: int64
1774


count    1774.000000
mean       19.747611
std        44.369541
min       -62.115000
25%       -14.234500
50%        14.067000
75%        51.434000
max       152.816000
Name: rpe1_fitness, dtype: float64

In [15]:
rpe1_gene_fitness = (
    rpe1_fitness[
        rpe1_fitness["OFFICIAL_SYMBOL"].isin(
            rpe1_targets
        )
    ][
        [
            "OFFICIAL_SYMBOL",
            "28-SCORE.1",
        ]
    ]
    .rename(
        columns={
            "OFFICIAL_SYMBOL": "gene",
            "28-SCORE.1": "fitness",
        }
    )
    .copy()
)

print(rpe1_gene_fitness.shape)
print(rpe1_gene_fitness["gene"].nunique())

rpe1_gene_fitness.to_csv(
    "../data/processed/rpe1_hart2015_gene_fitness.csv",
    index=False,
)

(1765, 2)
1765
